In [ ]:
!pip install google-adk google-genai nest_asyncio -q

import os, asyncio, re
import nest_asyncio
nest_asyncio.apply()

from getpass import getpass
from google.adk.agents import Agent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# API KEY
api_key = getpass("Enter API Key: ")
os.environ["GOOGLE_API_KEY"] = api_key

print("✅ Setup complete")

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


Enter API Key: ··········
✅ Setup complete


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search

library_main_agent = Agent(
    name="library_main_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Unified agent for library location, distance, opening hours, and availability in Myanmar cities.",

    instruction="""
You are the MASTER LIBRARY ASSISTANT 📚 for Myanmar.

You handle ALL library-related tasks:

-----------------------------------
INPUT TYPES YOU HANDLE:
-----------------------------------
1. Find libraries near user
2. Libraries within X km
3. Open/closed library today
4. Quiet/study libraries
5. Library opening hours & days
6. Library recommendations
7. Library + book availability queries

-----------------------------------
SUPPORTED CITIES:
-----------------------------------
- Mandalay (street grid system)
- Yangon (township-based)
- Naypyitaw (wide distances)
- Taunggyi (landmark-based)
- Bago (township-based)

-----------------------------------
DISTANCE RULE:
-----------------------------------
- Mandalay: 10 street blocks ≈ 1 km
- Yangon: township proximity ≈ 1–5 km
- Naypyitaw: distances are large (5–15 km typical)
- Others: estimate using landmarks

ONLY return libraries within requested km.

-----------------------------------
TASK FLOW:
-----------------------------------
1. Detect city from query
2. Estimate distance
3. Search libraries using Google Search
4. Filter ONLY within range
5. Check opening hours (today)
6. Check if open/closed today

-----------------------------------
OUTPUT FORMAT:
-----------------------------------

Library Name:
Address:
Distance (km):
Open Hours:
Days:
Today Status:
Why Recommended:

-----------------------------------
RULES:
-----------------------------------
- DO NOT include far libraries
- ALWAYS include distance
- If no match → say "No libraries found within range"
- Keep response clean and structured
"""
)

In [ ]:
book_main_agent = Agent(
    name="book_main_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Unified agent for book recommendation, review, search, and categorization.",

    instruction="""
You are the MASTER BOOK ASSISTANT 📖.

You handle ALL book-related tasks:

-----------------------------------
INPUT TYPES YOU HANDLE:
-----------------------------------
1. Book recommendations
2. Book reviews (analysis)
3. Book categories/types
4. Book search in libraries
5. Nearby book availability
6. Beginner / advanced classification

-----------------------------------
TASK ROUTING INSIDE YOUR LOGIC:
-----------------------------------

IF user says:
- "review" → perform deep book review
- "recommend" → suggest books
- "types" → classify book categories
- "search / find books" → locate books in libraries

-----------------------------------
BOOK REVIEW REQUIREMENTS:
-----------------------------------
Provide:
- Book Name
- Author
- Genre
- Level (Beginner/Intermediate/Advanced)
- Summary
- Key Lessons
- Pros
- Cons
- Rating (out of 5)
- Who should read

-----------------------------------
BOOK RECOMMENDATION:
-----------------------------------
Include:
- Level (Beginner / Intermediate / Advanced)
- Why recommended
- Practical value

-----------------------------------
BOOK TYPE CLASSIFICATION:
-----------------------------------
Categories:
- Technology (AI, Programming, Data Science)
- Business & Finance
- Self Development
- Psychology
- Science
- History
- Education
- Health

-----------------------------------
BOOK SEARCH (LIBRARY CONTEXT):
-----------------------------------
- Find where book exists nearby
- Estimate distance to library
- Check availability (likely / unknown)

-----------------------------------
OUTPUT FORMAT:
-----------------------------------

Book Name:
Author:
Category:
Level:
Summary:
Key Insight:
Recommendation Reason:
Availability (if searched):
Library (if found):

-----------------------------------
RULES:
-----------------------------------
- Be structured
- Be clear and short
- Do NOT mix unrelated tasks
- If multiple books → separate clearly
"""
)

In [ ]:
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
Return ONLY ONE label:

library → if query is about libraries, location, distance, opening hours
book → if query is about books, reviews, recommendations, types, search

RULE:
- No explanation
- Only one word output
"""
)

In [ ]:
agent_map = {
    "library": library_main_agent,
    "book": book_main_agent
}

In [ ]:
session_service = InMemorySessionService()
USER_ID = "user_001"

async def run_app(query):
    print(f"\n🗣️ Query: {query}")

    # -------------------------
    # STEP 1: ROUTER
    # -------------------------
    router_session = await session_service.create_session(
        app_name="router",
        user_id=USER_ID
    )

    router_runner = Runner(
        agent=router_agent,
        session_service=session_service,
        app_name="router"
    )

    route = ""

    async for event in router_runner.run_async(
        user_id=USER_ID,
        session_id=router_session.id,
        new_message=Content(parts=[Part(text=query)], role="user")
    ):
        if event.is_final_response():
            route = event.content.parts[0].text.strip()

    print(f"🧠 Route: {route}")

    # -------------------------
    # STEP 2: SELECT AGENT
    # -------------------------
    agent = agent_map.get(route)

    if not agent:
        print("❌ Invalid route")
        return

    # -------------------------
    # STEP 3: RUN SELECTED AGENT
    # -------------------------
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=USER_ID
    )

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user")
    ):
        if event.is_final_response():
            final = event.content.parts[0].text

    print("\n✅ RESULT:\n")
    print(final)

In [ ]:
await run_app("review Storytelling with Data book")


🗣️ Query: review Storytelling with Data book


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}

In [ ]:
await run_app("where can I get data analyst book near 62 107*108, Mandalay")

In [ ]:
await run_app("find library near Botahtaung Yangon within 5km open today")

**UI RUN**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import asyncio
import nest_asyncio

# Apply the patch to allow Colab to run async functions from buttons
nest_asyncio.apply()

# --- UI Components ---
user_prompt_input = widgets.Textarea(
    value='Find a quiet library near Rajpur Sonarpur.',
    placeholder='Ask for a library OR book recommendations...',
    description='🗣 Prompt:',
    layout=widgets.Layout(width='80%', height='60px')
)

btn_ask = widgets.Button(
    description='🤖 Ask the Router',
    button_style='primary',
    layout=widgets.Layout(width='auto')
)

output_area = widgets.Output()

# --- Async UI Bridge to your run_app ---
async def process_user_request():
    query = user_prompt_input.value

    with output_area:
        print(f"-> Sending request to Router: '{query}'")
        print("-> 🧭 Router is deciding which agent to use...")

    # Call your router function!
    response_text = await run_app(query)

    # Display the final routed result
    with output_area:
        clear_output()
        display(Markdown(response_text))

# --- Sync Click Handler ---
def on_ask_clicked(btn):
    # Lock the UI
    btn.description = "🧭 Routing..."
    btn.button_style = "warning"
    btn.disabled = True

    with output_area:
        clear_output()

    try:
        loop = asyncio.get_event_loop()
        loop.run_until_complete(process_user_request())
    except Exception as e:
        with output_area:
            print(f"🛑 Execution Error: {e}")
    finally:
        # Unlock the UI
        btn.description = "🤖 Ask the Router"
        btn.button_style = "primary"
        btn.disabled = False

# Bind the button
btn_ask.on_click(on_ask_clicked)

# --- Render the UI ---
ui_dashboard = widgets.VBox([
    widgets.HTML("<h2>🧠 Smart Library & Literature Router</h2>"),
    user_prompt_input,
    btn_ask,
    output_area
])

display(ui_dashboard)


🗣️ Query: မန္တလေးအနီးတစ်ဝိုက် library များကို ဖော်ပြပါ
